In [2]:
import os
import time
import yaml
import cv2
import pandas as pd
import numpy as np
import torch
from pathlib import Path
from ultralytics import YOLO
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.optimize import linear_sum_assignment

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.6.0+cu124
CUDA available: True


In [6]:
DATA_DIR = r"C:\Users\ASUS\Documents\GitHub\IUT\MainProjectOfUniversity\data\data"

def check_structure(base_dir):
    expected = [r"images\train", r"images\val", r"labels\train", r"labels\val"]
    for sub in expected:
        p = Path(base_dir) / sub
        exists = p.exists()
        n_files = len(list(p.glob("*"))) if exists else 0
        print(f"{sub:20s} exists={exists}  files={n_files}")
        
check_structure(DATA_DIR)

images\train         exists=True  files=1035
images\val           exists=True  files=452
labels\train         exists=True  files=1036
labels\val           exists=True  files=452


In [7]:
data_yaml = {
    "path": os.path.abspath(DATA_DIR),
    "train": "images/train",
    "val": "images/val",
    "nc": 1,
    "names": ["Pollo"]
}

yaml_path = os.path.join(DATA_DIR, "pio_data.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(f"data.yaml written to: {yaml_path}\n")
print(yaml.dump(data_yaml, sort_keys=False))

data.yaml written to: C:\Users\ASUS\Documents\GitHub\IUT\MainProjectOfUniversity\data\data\pio_data.yaml

path: C:\Users\ASUS\Documents\GitHub\IUT\MainProjectOfUniversity\data\data
train: images/train
val: images/val
nc: 1
names:
- Pollo



In [8]:
sample_labels = list(Path(DATA_DIR, "labels/train").glob("*.txt"))[:3]

for lbl_file in sample_labels:
    print(f"--- {lbl_file.name} ---")
    with open(lbl_file) as f:
        lines = f.readlines()
    print(f"  {len(lines)} annotated chickens")
    for line in lines[:3]:
        cls, xc, yc, w, h = line.strip().split()
        print(f"  class={cls} x_center={xc} y_center={yc} w={w} h={h}")

--- C-W1-0001.txt ---
  741 annotated chickens
  class=0 x_center=0.435417 y_center=0.485185 w=0.023958 h=0.048148
  class=0 x_center=0.359115 y_center=0.455556 w=0.020313 h=0.050000
  class=0 x_center=0.544271 y_center=0.554167 w=0.020833 h=0.052778
--- C-W1-0002.txt ---
  616 annotated chickens
  class=0 x_center=0.520833 y_center=0.533333 w=0.023958 h=0.044444
  class=0 x_center=0.665365 y_center=0.261574 w=0.022396 h=0.045370
  class=0 x_center=0.590365 y_center=0.527315 w=0.027604 h=0.037963
--- C-W1-0003.txt ---
  175 annotated chickens
  class=0 x_center=0.105892 y_center=0.247048 w=0.0189926 h=0.0368801
  class=0 x_center=0.159317 y_center=0.161137 w=0.0177709 h=0.0393683
  class=0 x_center=0.128776 y_center=0.852044 w=0.0182641 h=0.0272009


In [10]:
model = YOLO("yolov10m.pt")  

In [11]:
results = model.train(
    data=yaml_path,
    imgsz=640,
    epochs=100,
    batch=4,
    optimizer="AdamW",
    lr0=0.002,
    momentum=0.9,
    project="PIO_runs",
    name="yolov10m_chicken_detect",
    patience=20,
    verbose=True
)

Ultralytics 8.4.123  Python-3.12.4 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050, 8191MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\ASUS\Documents\GitHub\IUT\MainProjectOfUniversity\data\data\pio_data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10m.pt, momentum=0.9, mo

In [12]:
metrics = model.val(data=yaml_path, imgsz=640)

print(f"Precision : {metrics.box.mp:.3f}")
print(f"Recall    : {metrics.box.mr:.3f}")
print(f"mAP@50    : {metrics.box.map50:.3f}")
print(f"mAP@50-95 : {metrics.box.map:.3f}")

Ultralytics 8.4.123  Python-3.12.4 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050, 8191MiB)
YOLOv10m summary (fused): 136 layers, 15,313,747 parameters, 0 gradients, 59.0 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1681.696.1 MB/s, size: 1002.8 KB)
val: Scanning C:\Users\ASUS\Documents\GitHub\IUT\MainProjectOfUniversity\data\data\labels\val.cache... 452 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 452/452  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 2.0it/s 14.7s0.2s
                   all        452      73859      0.953      0.877      0.897      0.684
Speed: 0.9ms preprocess, 12.5ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to C:\Users\ASUS\Documents\GitHub\IUT\runs\detect\val
Precision : 0.953
Recall    : 0.877
mAP@50    : 0.897
mAP@50-95 : 0.684


In [13]:
def count_chickens(image_path, model, conf_threshold=0.25):
    results = model.predict(
        source=image_path,
        conf=conf_threshold,
        imgsz=640,
        save=True,
        project="PIO_runs",
        name="predictions",
        exist_ok=True
    )
    result = results[0]
    num_chickens = len(result.boxes)
    print(f"Image: {image_path}")
    print(f"Detected chickens: {num_chickens}")
    return num_chickens, result

val_dir = os.path.join(DATA_DIR, "images/val")
test_image = os.path.join(val_dir, os.listdir(val_dir)[0])
count, result = count_chickens(test_image, model)


image 1/1 C:\Users\ASUS\Documents\GitHub\IUT\MainProjectOfUniversity\data\data\images\val\C-W1-V0001.jpg: 384x640 300 Pollos, 20.7ms
Speed: 4.6ms preprocess, 20.7ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)
Results saved to C:\Users\ASUS\Documents\GitHub\IUT\runs\detect\PIO_runs\predictions
Image: C:\Users\ASUS\Documents\GitHub\IUT\MainProjectOfUniversity\data\data\images/val\C-W1-V0001.jpg
Detected chickens: 300


In [14]:
def count_chickens_in_folder(folder_path, model, conf_threshold=0.25):
    counts = {}
    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

    for img_name in image_files:
        img_path = os.path.join(folder_path, img_name)
        results = model.predict(source=img_path, conf=conf_threshold, imgsz=640, verbose=False)
        counts[img_name] = len(results[0].boxes)

    return counts

all_counts = count_chickens_in_folder(val_dir, model)

for img_name, n in list(all_counts.items())[:10]:
    print(f"{img_name}: {n} chickens")

avg_count = sum(all_counts.values()) / len(all_counts)
print(f"\nAverage chickens per image: {avg_count:.1f}")

C-W1-V0001.jpg: 300 chickens
C-W1-V0002.jpg: 300 chickens
C-W1-V0003.jpg: 122 chickens
C-W1-V0004.jpg: 161 chickens
C-W1-V0005.jpg: 143 chickens
C-W1-V0006.jpg: 187 chickens
C-W1-V0007.jpg: 124 chickens
C-W1-V0008.jpg: 79 chickens
C-W1-V0009.jpg: 53 chickens
C-W1-V0010.jpg: 300 chickens

Average chickens per image: 155.6


In [3]:
DATA_DIR = r"C:\Users\ASUS\Documents\GitHub\IUT\MainProjectOfUniversity\data\data"
VIDEO_DIR = r"C:\Users\ASUS\Documents\GitHub\IUT\MainProjectOfUniversity\data\Videos 2023"
MODEL_PATH = r"C:\Users\ASUS\Documents\GitHub\IUT\runs\detect\PIO_runs\yolov10m_chicken_detect\weights\best.pt"

FPS = 30                  # original video fps
VID_STRIDE = 3             # process every 3rd frame -> effective 10 fps
MOVING_SPEED_THRESHOLD = 10.0
MIN_TRACK_LENGTH = 20

MAX_GAP_SECONDS = 2.0
STITCH_DISTANCE_MULTIPLIER = 3.0

video_week_map = {
    "0004": 1,
    "0006": 2,
    "0007": 3,
    "0008": 4,
    "0013": 5
}

device = 0 if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: 0


In [4]:
track_model = YOLO(MODEL_PATH)
device = 0 if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: 0


In [4]:
track_model = YOLO(MODEL_PATH)
print("Model loaded.")

Model loaded.


In [8]:
import ultralytics
from pathlib import Path
import shutil
import yaml

# Locate the real default botsort.yaml shipped with your installed ultralytics version
ultralytics_path = Path(ultralytics.__file__).parent
default_botsort_path = ultralytics_path / "cfg" / "trackers" / "botsort.yaml"

print(f"Found default config at: {default_botsort_path}")
print(f"Exists: {default_botsort_path.exists()}")

# Load the real default config as our starting point
with open(default_botsort_path, "r") as f:
    custom_botsort_config = yaml.safe_load(f)

print("\nDefault config contents:")
print(yaml.dump(custom_botsort_config))

# Only override the values we actually want to change
custom_botsort_config["track_buffer"] = 90     # increased from default (usually 30)
custom_botsort_config["match_thresh"] = 0.8

CUSTOM_TRACKER_PATH = "custom_botsort.yaml"
with open(CUSTOM_TRACKER_PATH, "w") as f:
    yaml.dump(custom_botsort_config, f)

print(f"\nSaved updated config to: {CUSTOM_TRACKER_PATH}")
print(yaml.dump(custom_botsort_config))

Found default config at: C:\Users\ASUS\Documents\GitHub\IUT\MainProjectOfUniversity\venv\Lib\site-packages\ultralytics\cfg\trackers\botsort.yaml
Exists: True

Default config contents:
appearance_thresh: 0.8
fuse_score: true
gmc_method: sparseOptFlow
match_thresh: 0.8
model: auto
new_track_thresh: 0.25
proximity_thresh: 0.5
track_buffer: 30
track_high_thresh: 0.25
track_low_thresh: 0.1
tracker_type: botsort
with_reid: false


Saved updated config to: custom_botsort.yaml
appearance_thresh: 0.8
fuse_score: true
gmc_method: sparseOptFlow
match_thresh: 0.8
model: auto
new_track_thresh: 0.25
proximity_thresh: 0.5
track_buffer: 90
track_high_thresh: 0.25
track_low_thresh: 0.1
tracker_type: botsort
with_reid: false



In [9]:
def track_video_strided_with_video(video_path, model, tracker=CUSTOM_TRACKER_PATH,
                                     conf=0.25, device=0, vid_stride=VID_STRIDE,
                                     output_video_path=None):
    cap = cv2.VideoCapture(video_path)
    orig_fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    effective_fps = orig_fps / vid_stride

    writer = None
    if output_video_path:
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(output_video_path, fourcc, effective_fps, (width, height))

    results = model.track(
        source=video_path,
        tracker=tracker,
        conf=conf,
        persist=True,
        device=device,
        save=False,
        verbose=False,
        stream=True,
        vid_stride=vid_stride
    )

    frame_records = []
    for frame_idx, result in enumerate(results):
        if result.boxes.id is not None:
            ids = result.boxes.id.cpu().numpy().astype(int)
            xywh = result.boxes.xywh.cpu().numpy()

            for track_id, (x, y, w, h) in zip(ids, xywh):
                frame_records.append({
                    "frame": frame_idx * vid_stride,
                    "track_id": track_id,
                    "x": x, "y": y, "w": w, "h": h
                })

        if writer is not None:
            annotated_frame = result.plot()
            writer.write(annotated_frame)

    if writer is not None:
        writer.release()

    return pd.DataFrame(frame_records)

In [10]:
RAW_TRAJECTORY_DIR = "trajectories_raw_v2"
ANNOTATED_VIDEO_DIR = "annotated_videos"
os.makedirs(RAW_TRAJECTORY_DIR, exist_ok=True)
os.makedirs(ANNOTATED_VIDEO_DIR, exist_ok=True)

for video_name in video_week_map:
    video_path = os.path.join(VIDEO_DIR, f"{video_name}.mp4")
    output_video_path = os.path.join(ANNOTATED_VIDEO_DIR, f"{video_name}_annotated.mp4")
    print(f"Tracking {video_name} (saving CSV + annotated video)...")

    start = time.time()
    df = track_video_strided_with_video(
        video_path, track_model,
        device=device, vid_stride=VID_STRIDE,
        output_video_path=output_video_path
    )
    elapsed = time.time() - start

    out_csv_path = os.path.join(RAW_TRAJECTORY_DIR, f"{video_name}_trajectories.csv")
    df.to_csv(out_csv_path, index=False)
    print(f"  Saved CSV: {out_csv_path} ({df['track_id'].nunique()} raw IDs)")
    print(f"  Saved video: {output_video_path}")
    print(f"  Time: {elapsed/60:.1f} min\n")

print("All videos tracked. CSVs and annotated videos both saved.")

Tracking 0004 (saving CSV + annotated video)...
  Saved CSV: trajectories_raw_v2\0004_trajectories.csv (14295 raw IDs)
  Saved video: annotated_videos\0004_annotated.mp4
  Time: 92.4 min

Tracking 0006 (saving CSV + annotated video)...
  Saved CSV: trajectories_raw_v2\0006_trajectories.csv (58200 raw IDs)
  Saved video: annotated_videos\0006_annotated.mp4
  Time: 155.9 min

Tracking 0007 (saving CSV + annotated video)...
  Saved CSV: trajectories_raw_v2\0007_trajectories.csv (31283 raw IDs)
  Saved video: annotated_videos\0007_annotated.mp4
  Time: 182.9 min

Tracking 0008 (saving CSV + annotated video)...
  Saved CSV: trajectories_raw_v2\0008_trajectories.csv (8345 raw IDs)
  Saved video: annotated_videos\0008_annotated.mp4
  Time: 69.7 min

Tracking 0013 (saving CSV + annotated video)...
  Saved CSV: trajectories_raw_v2\0013_trajectories.csv (82206 raw IDs)
  Saved video: annotated_videos\0013_annotated.mp4
  Time: 168.8 min

All videos tracked. CSVs and annotated videos both saved.


In [19]:
def estimate_velocity(track_df, n_points=3):
    if len(track_df) < 2:
        return 0.0, 0.0
    pts = track_df.tail(n_points) if len(track_df) >= n_points else track_df
    dx = pts["x"].iloc[-1] - pts["x"].iloc[0]
    dy = pts["y"].iloc[-1] - pts["y"].iloc[0]
    dframe = pts["frame"].iloc[-1] - pts["frame"].iloc[0]
    if dframe == 0:
        return 0.0, 0.0
    return dx / dframe, dy / dframe


def stitch_tracks(df, fps=FPS, max_gap_seconds=MAX_GAP_SECONDS, distance_multiplier=STITCH_DISTANCE_MULTIPLIER):
    max_gap_frames = max_gap_seconds * fps
    avg_width = df["w"].mean()
    max_distance = avg_width * distance_multiplier

    track_ends, track_starts = [], []
    for track_id, track_df in df.groupby("track_id"):
        track_df = track_df.sort_values("frame")
        vx, vy = estimate_velocity(track_df)
        track_ends.append({"track_id": track_id, "end_frame": track_df["frame"].iloc[-1],
                            "x": track_df["x"].iloc[-1], "y": track_df["y"].iloc[-1], "vx": vx, "vy": vy})
        track_starts.append({"track_id": track_id, "start_frame": track_df["frame"].iloc[0],
                              "x": track_df["x"].iloc[0], "y": track_df["y"].iloc[0]})

    ends_df = pd.DataFrame(track_ends).reset_index(drop=True)
    starts_df = pd.DataFrame(track_starts).sort_values("start_frame").reset_index(drop=True)

    n_ends, n_starts = len(ends_df), len(starts_df)
    start_frames_array = starts_df["start_frame"].values

    # ---- collect all valid candidate edges (end_idx, start_idx, distance) ----
    edges = []
    for i, end_row in ends_df.iterrows():
        lo = np.searchsorted(start_frames_array, end_row["end_frame"], side="right")
        hi = np.searchsorted(start_frames_array, end_row["end_frame"] + max_gap_frames, side="right")

        for j in range(lo, hi):
            start_row = starts_df.iloc[j]
            if start_row["track_id"] == end_row["track_id"]:
                continue

            dt = start_row["start_frame"] - end_row["end_frame"]
            pred_x = end_row["x"] + end_row["vx"] * dt
            pred_y = end_row["y"] + end_row["vy"] * dt
            dist = np.sqrt((pred_x - start_row["x"])**2 + (pred_y - start_row["y"])**2)

            if dist <= max_distance:
                edges.append((dist, i, j))

    print(f"  {n_ends} ends, {n_starts} starts -> {len(edges)} candidate edges within threshold")

    # ---- greedy matching: take best (smallest-distance) edges first, skip if either side already used ----
    edges.sort(key=lambda e: e[0])   # ascending by distance -> best matches claimed first

    used_ends = set()
    used_starts = set()
    matches = []

    for dist, i, j in edges:
        if i in used_ends or j in used_starts:
            continue
        used_ends.add(i)
        used_starts.add(j)
        matches.append((i, j))

    # ---- union-find to merge chains of matched tracks ----
    parent = {tid: tid for tid in df["track_id"].unique()}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(x, y):
        parent[find(x)] = find(y)

    for i, j in matches:
        union(ends_df.iloc[i]["track_id"], starts_df.iloc[j]["track_id"])

    df = df.copy()
    df["stitched_track_id"] = df["track_id"].map(lambda tid: find(tid))
    print(f"  Stitching merged {len(matches)} pairs. IDs: {df['track_id'].nunique()} -> {df['stitched_track_id'].nunique()}")
    return df

In [20]:
STITCHED_TRAJECTORY_DIR = "trajectories_stitched"
os.makedirs(STITCHED_TRAJECTORY_DIR, exist_ok=True)

for video_name in video_week_map:
    raw_path = os.path.join(RAW_TRAJECTORY_DIR, f"{video_name}_trajectories.csv")
    df = pd.read_csv(raw_path)

    print(f"Stitching {video_name}...")
    stitched_df = stitch_tracks(df)
    stitched_df["track_id"] = stitched_df["stitched_track_id"]
    stitched_df = stitched_df.drop(columns=["stitched_track_id"])

    out_path = os.path.join(STITCHED_TRAJECTORY_DIR, f"{video_name}_trajectories.csv")
    stitched_df.to_csv(out_path, index=False)
    print(f"  Saved: {out_path}\n")

TRAJECTORY_DIR = STITCHED_TRAJECTORY_DIR

Stitching 0004...
  14295 ends, 14295 starts -> 22237 candidate edges within threshold
  Stitching merged 8637 pairs. IDs: 14295 -> 5658
  Saved: trajectories_stitched\0004_trajectories.csv

Stitching 0006...
  58200 ends, 58200 starts -> 81749 candidate edges within threshold
  Stitching merged 31188 pairs. IDs: 58200 -> 27012
  Saved: trajectories_stitched\0006_trajectories.csv

Stitching 0007...
  31283 ends, 31283 starts -> 60538 candidate edges within threshold
  Stitching merged 15792 pairs. IDs: 31283 -> 15491
  Saved: trajectories_stitched\0007_trajectories.csv

Stitching 0008...
  8345 ends, 8345 starts -> 15079 candidate edges within threshold
  Stitching merged 4209 pairs. IDs: 8345 -> 4136
  Saved: trajectories_stitched\0008_trajectories.csv

Stitching 0013...
  82206 ends, 82206 starts -> 143216 candidate edges within threshold
  Stitching merged 42549 pairs. IDs: 82206 -> 39657
  Saved: trajectories_stitched\0013_trajectories.csv



In [21]:
def filter_short_tracks(df, min_length=MIN_TRACK_LENGTH):
    track_lengths = df.groupby("track_id").size()
    valid_ids = track_lengths[track_lengths >= min_length].index
    return df[df["track_id"].isin(valid_ids)].copy()

for video_name in video_week_map:
    csv_path = os.path.join(TRAJECTORY_DIR, f"{video_name}_trajectories.csv")
    df = pd.read_csv(csv_path)
    filtered = filter_short_tracks(df)
    print(f"{video_name}: {df['track_id'].nunique()} -> {filtered['track_id'].nunique()} tracks after filtering")

0004: 5658 -> 3170 tracks after filtering
0006: 27012 -> 14902 tracks after filtering
0007: 15491 -> 10331 tracks after filtering
0008: 4136 -> 2416 tracks after filtering
0013: 39657 -> 25974 tracks after filtering


In [22]:
def compute_basic_params(track_df, fps=FPS, moving_thresh=MOVING_SPEED_THRESHOLD):
    track_df = track_df.sort_values("frame").reset_index(drop=True)
    n_points = len(track_df)
    if n_points < 2:
        return None

    x = track_df["x"].values
    y = track_df["y"].values
    frames = track_df["frame"].values

    dx = np.diff(x)
    dy = np.diff(y)
    dt = np.diff(frames) / fps
    dt[dt == 0] = np.nan

    step_distance = np.sqrt(dx**2 + dy**2)
    step_speed = step_distance / dt

    total_distance = np.nansum(step_distance)
    avg_speed = np.nanmean(step_speed)
    speed_std = np.nanstd(step_speed)
    max_speed = np.nanmax(step_speed)

    if len(step_speed) >= 2:
        acceleration = np.diff(step_speed) / dt[1:]
        avg_acceleration = np.nanmean(np.abs(acceleration))
        accel_std = np.nanstd(acceleration)
    else:
        avg_acceleration, accel_std = np.nan, np.nan

    moving_mask = step_speed > moving_thresh
    movement_ratio = np.nanmean(moving_mask)

    return {
        "n_points": n_points,
        "duration_sec": (frames[-1] - frames[0]) / fps,
        "total_distance_px": total_distance,
        "avg_speed_px_s": avg_speed,
        "speed_std_px_s": speed_std,
        "max_speed_px_s": max_speed,
        "avg_acceleration_px_s2": avg_acceleration,
        "acceleration_std": accel_std,
        "movement_ratio": movement_ratio,
        "rest_ratio": 1 - movement_ratio
    }

In [23]:
all_results = []

for video_name, week_num in video_week_map.items():
    csv_path = os.path.join(TRAJECTORY_DIR, f"{video_name}_trajectories.csv")
    df = pd.read_csv(csv_path)
    df = filter_short_tracks(df)
    print(f"Processing {video_name} (week {week_num}) — {df['track_id'].nunique()} tracks")

    for track_id, track_df in df.groupby("track_id"):
        params = compute_basic_params(track_df)
        if params is None:
            continue
        params.update({"video": video_name, "week": week_num, "track_id": track_id})
        all_results.append(params)

results_df = pd.DataFrame(all_results)
cols = ["week", "video", "track_id", "n_points", "duration_sec", "total_distance_px",
        "avg_speed_px_s", "speed_std_px_s", "max_speed_px_s",
        "avg_acceleration_px_s2", "acceleration_std", "movement_ratio", "rest_ratio"]
results_df = results_df[cols]
print(f"\nTotal rows: {len(results_df)}")
results_df.head(10)

Processing 0004 (week 1) — 3170 tracks
Processing 0006 (week 2) — 14902 tracks
Processing 0007 (week 3) — 10331 tracks
Processing 0008 (week 4) — 2416 tracks
Processing 0013 (week 5) — 25974 tracks

Total rows: 56793


,week,video,track_id,n_points,duration_sec,total_distance_px,avg_speed_px_s,speed_std_px_s,max_speed_px_s,avg_acceleration_px_s2,acceleration_std,movement_ratio,rest_ratio
0,1,0004,6.0,1686,170.4,1563.891467,9.165333,10.356686,101.423494,57.214289,94.212555,0.291395,0.708605
1,1,0004,11.0,1749,181.3,1440.720764,8.049617,8.695377,97.774539,51.712240,87.503190,0.254005,0.745995
2,1,0004,12.0,451,53.2,895.682896,18.133039,20.111909,156.091815,121.373645,191.048605,0.551111,0.448889
3,1,0004,13.0,591,59.5,819.215795,13.697798,17.126496,266.930732,79.174960,134.895730,0.474576,0.525424
4,1,0004,17.0,216,21.5,666.302678,30.990822,31.694809,134.441444,111.873343,167.074077,0.609302,0.390698
5,1,0004,20.0,592,59.3,615.548785,10.398375,14.335720,152.580131,55.809235,88.439317,0.302876,0.697124
6,1,0004,21.0,2193,236.6,3608.961429,15.487333,23.093233,236.190389,79.583285,140.926827,0.383668,0.616332
7,1,0004,22.0,258,25.7,428.974103,16.691599,16.716387,88.937011,104.446205,154.604926,0.505837,0.494163
8,1,0004,23.0,366,36.5,289.566214,7.933321,6.431369,48.812769,55.586874,80.188300,0.287671,0.712329
9,1,0004,25.0,38,6.5,176.976793,34.757007,28.833878,134.868404,264.742029,329.243289,0.783784,0.216216


In [24]:
def compute_bouts(track_df, fps=FPS, moving_thresh=MOVING_SPEED_THRESHOLD):
    track_df = track_df.sort_values("frame").reset_index(drop=True)
    n_points = len(track_df)
    if n_points < 3:
        return None

    x = track_df["x"].values
    y = track_df["y"].values
    frames = track_df["frame"].values

    dx = np.diff(x)
    dy = np.diff(y)
    dt = np.diff(frames) / fps
    dt[dt == 0] = np.nan

    step_speed = np.sqrt(dx**2 + dy**2) / dt
    is_moving = np.nan_to_num(step_speed > moving_thresh, nan=0).astype(bool)
    step_times = dt

    movement_bout_durations, rest_bout_durations = [], []
    rest_bout_end_positions, rest_bout_end_times = [], []

    cumulative_time = step_times[0] if not np.isnan(step_times[0]) else 0
    current_state = is_moving[0]
    current_duration = cumulative_time

    for i in range(1, len(is_moving)):
        state = is_moving[i]
        dur = step_times[i] if not np.isnan(step_times[i]) else 0
        cumulative_time += dur

        if state == current_state:
            current_duration += dur
        else:
            if current_state:
                movement_bout_durations.append(current_duration)
            else:
                rest_bout_durations.append(current_duration)
                rest_bout_end_positions.append((x[i], y[i]))
                rest_bout_end_times.append(cumulative_time)
            current_state = state
            current_duration = dur

    if current_state:
        movement_bout_durations.append(current_duration)
    else:
        rest_bout_durations.append(current_duration)
        rest_bout_end_positions.append((x[-1], y[-1]))
        rest_bout_end_times.append(cumulative_time)

    inter_rest_distances = [
        np.sqrt((rest_bout_end_positions[i][0]-rest_bout_end_positions[i-1][0])**2 +
                (rest_bout_end_positions[i][1]-rest_bout_end_positions[i-1][1])**2)
        for i in range(1, len(rest_bout_end_positions))
    ]
    inter_rest_time_gaps = [
        rest_bout_end_times[i] - rest_bout_end_times[i-1]
        for i in range(1, len(rest_bout_end_times))
    ]

    return {
        "n_movement_bouts": len(movement_bout_durations),
        "n_rest_bouts": len(rest_bout_durations),
        "avg_movement_bout_sec": np.mean(movement_bout_durations) if movement_bout_durations else np.nan,
        "max_movement_bout_sec": np.max(movement_bout_durations) if movement_bout_durations else np.nan,
        "avg_rest_bout_sec": np.mean(rest_bout_durations) if rest_bout_durations else np.nan,
        "max_rest_bout_sec": np.max(rest_bout_durations) if rest_bout_durations else np.nan,
        "avg_distance_between_rests_px": np.mean(inter_rest_distances) if inter_rest_distances else np.nan,
        "avg_time_between_rests_sec": np.mean(inter_rest_time_gaps) if inter_rest_time_gaps else np.nan,
    }

In [25]:
bout_results = []

for video_name, week_num in video_week_map.items():
    csv_path = os.path.join(TRAJECTORY_DIR, f"{video_name}_trajectories.csv")
    df = pd.read_csv(csv_path)
    df = filter_short_tracks(df)

    for track_id, track_df in df.groupby("track_id"):
        bp = compute_bouts(track_df)
        if bp is None:
            continue
        bp.update({"video": video_name, "week": week_num, "track_id": track_id})
        bout_results.append(bp)

bouts_df = pd.DataFrame(bout_results)
cols = ["week", "video", "track_id", "n_movement_bouts", "n_rest_bouts",
        "avg_movement_bout_sec", "max_movement_bout_sec",
        "avg_rest_bout_sec", "max_rest_bout_sec",
        "avg_distance_between_rests_px", "avg_time_between_rests_sec"]
bouts_df = bouts_df[cols]

merged_df = results_df.merge(bouts_df, on=["week", "video", "track_id"], how="outer")
merged_df.to_csv("chicken_movement_params_full.csv", index=False)
print(f"Saved. Shape: {merged_df.shape}")

Saved. Shape: (56793, 21)


In [26]:
def compute_direction_and_turns(track_df, fps=FPS, moving_thresh=MOVING_SPEED_THRESHOLD, sharp_turn_angle_deg=45.0):
    track_df = track_df.sort_values("frame").reset_index(drop=True)
    if len(track_df) < 3:
        return None

    x, y, frames = track_df["x"].values, track_df["y"].values, track_df["frame"].values
    dx, dy = np.diff(x), np.diff(y)
    dt = np.diff(frames) / fps
    dt[dt == 0] = np.nan

    step_speed = np.sqrt(dx**2 + dy**2) / dt
    is_moving = step_speed > moving_thresh
    step_angle = np.degrees(np.arctan2(dy, dx))

    moving_angles = step_angle[is_moving]
    if len(moving_angles) > 0:
        rad = np.radians(moving_angles)
        mean_sin, mean_cos = np.mean(np.sin(rad)), np.mean(np.cos(rad))
        dominant_direction_deg = np.degrees(np.arctan2(mean_sin, mean_cos))
        direction_consistency = np.sqrt(mean_sin**2 + mean_cos**2)
    else:
        dominant_direction_deg, direction_consistency = np.nan, np.nan

    sharp_turn_count, valid_transitions = 0, 0
    for i in range(1, len(step_angle)):
        if is_moving[i] and is_moving[i-1]:
            valid_transitions += 1
            diff = (step_angle[i] - step_angle[i-1] + 180) % 360 - 180
            if abs(diff) >= sharp_turn_angle_deg:
                sharp_turn_count += 1

    return {
        "dominant_direction_deg": dominant_direction_deg,
        "direction_consistency": direction_consistency,
        "n_sharp_turns": sharp_turn_count,
        "sharp_turn_rate": sharp_turn_count / valid_transitions if valid_transitions > 0 else np.nan,
    }

In [27]:
direction_results = []

for video_name, week_num in video_week_map.items():
    csv_path = os.path.join(TRAJECTORY_DIR, f"{video_name}_trajectories.csv")
    df = pd.read_csv(csv_path)
    df = filter_short_tracks(df)

    for track_id, track_df in df.groupby("track_id"):
        dp = compute_direction_and_turns(track_df)
        if dp is None:
            continue
        dp.update({"video": video_name, "week": week_num, "track_id": track_id})
        direction_results.append(dp)

direction_df = pd.DataFrame(direction_results)
cols = ["week", "video", "track_id", "dominant_direction_deg", "direction_consistency", "n_sharp_turns", "sharp_turn_rate"]
direction_df = direction_df[cols]

merged_df = merged_df.merge(direction_df, on=["week", "video", "track_id"], how="outer")
merged_df.to_csv("chicken_movement_params_full.csv", index=False)
print(f"Saved. Shape: {merged_df.shape}")

Saved. Shape: (56793, 25)


In [28]:
def compute_grouping_per_frame(frame_df, group_distance_threshold):
    ids = frame_df["track_id"].values
    coords = frame_df[["x", "y"]].values
    n = len(ids)

    if n == 1:
        return pd.DataFrame([{"track_id": ids[0], "group_size": 1, "is_alone": True, "nearest_neighbor_dist": np.nan}])

    tree = cKDTree(coords)
    dists, _ = tree.query(coords, k=2)
    nearest_neighbor_dist = dists[:, 1]

    pairs = tree.query_pairs(r=group_distance_threshold, output_type="ndarray")
    if len(pairs) > 0:
        row = np.concatenate([pairs[:, 0], pairs[:, 1]])
        col = np.concatenate([pairs[:, 1], pairs[:, 0]])
        adjacency = coo_matrix((np.ones(len(row)), (row, col)), shape=(n, n))
        _, labels = connected_components(adjacency, directed=False)
    else:
        labels = np.arange(n)

    group_sizes = pd.Series(labels).map(pd.Series(labels).value_counts()).values

    return pd.DataFrame({
        "track_id": ids, "group_size": group_sizes,
        "is_alone": group_sizes == 1, "nearest_neighbor_dist": nearest_neighbor_dist
    })


def compute_grouping_for_video(df, group_distance_threshold=None):
    if group_distance_threshold is None:
        group_distance_threshold = 2 * df["w"].mean()

    all_frame_results = []
    for frame_id, frame_df in df.groupby("frame"):
        r = compute_grouping_per_frame(frame_df[["track_id", "x", "y"]], group_distance_threshold)
        r["frame"] = frame_id
        all_frame_results.append(r)

    per_frame_df = pd.concat(all_frame_results, ignore_index=True)
    per_frame_df["dist_when_alone"] = np.where(per_frame_df["is_alone"], per_frame_df["nearest_neighbor_dist"], np.nan)

    return per_frame_df.groupby("track_id").agg(
        avg_nearest_neighbor_dist=("nearest_neighbor_dist", "mean"),
        avg_group_size=("group_size", "mean"),
        frac_frames_alone=("is_alone", "mean"),
        avg_dist_to_nearest_group_when_alone=("dist_when_alone", "mean")
    ).reset_index()

In [29]:
grouping_results = []

for video_name, week_num in video_week_map.items():
    csv_path = os.path.join(TRAJECTORY_DIR, f"{video_name}_trajectories.csv")
    df = pd.read_csv(csv_path)
    df = filter_short_tracks(df)
    print(f"Grouping {video_name} (week {week_num})...")

    summary = compute_grouping_for_video(df)
    summary["video"] = video_name
    summary["week"] = week_num
    grouping_results.append(summary)

grouping_df = pd.concat(grouping_results, ignore_index=True)
cols = ["week", "video", "track_id", "avg_nearest_neighbor_dist", "avg_group_size", "frac_frames_alone", "avg_dist_to_nearest_group_when_alone"]
grouping_df = grouping_df[cols]

merged_df = merged_df.merge(grouping_df, on=["week", "video", "track_id"], how="outer")
merged_df.to_csv("chicken_movement_params_full.csv", index=False)
print(f"Final saved. Shape: {merged_df.shape}")

Grouping 0004 (week 1)...
Grouping 0006 (week 2)...
Grouping 0007 (week 3)...
Grouping 0008 (week 4)...
Grouping 0013 (week 5)...
Final saved. Shape: (56793, 29)


In [30]:
print("Final usable track segments per week:")
print(merged_df.groupby("week")["track_id"].count())
merged_df.head(10)

Final usable track segments per week:
week
1     3170
2    14902
3    10331
4     2416
5    25974
Name: track_id, dtype: int64


,week,video,track_id,n_points,duration_sec,total_distance_px,avg_speed_px_s,speed_std_px_s,max_speed_px_s,avg_acceleration_px_s2,...,avg_distance_between_rests_px,avg_time_between_rests_sec,dominant_direction_deg,direction_consistency,n_sharp_turns,sharp_turn_rate,avg_nearest_neighbor_dist,avg_group_size,frac_frames_alone,avg_dist_to_nearest_group_when_alone
0,1,0004,6.0,1686,170.4,1563.891467,9.165333,10.356686,101.423494,57.214289,...,3.870264,0.820098,-110.395117,0.082408,96,0.335664,57.058126,77.402135,0.00000,NaN
1,1,0004,11.0,1749,181.3,1440.720764,8.049617,8.695377,97.774539,51.712240,...,3.082174,0.883415,178.355966,0.057320,99,0.414226,44.863991,77.284734,0.00000,NaN
2,1,0004,12.0,451,53.2,895.682896,18.133039,20.111909,156.091815,121.373645,...,5.484899,0.603704,-168.885251,0.082624,68,0.412121,49.832924,78.800443,0.00000,NaN
3,1,0004,13.0,591,59.5,819.215795,13.697798,17.126496,266.930732,79.174960,...,4.752629,0.681176,-66.087305,0.101486,69,0.357513,56.740154,78.834179,0.00000,NaN
4,1,0004,17.0,216,21.5,666.302678,30.990822,31.694809,134.441444,111.873343,...,16.983930,0.905000,117.184648,0.236334,22,0.200000,74.129933,77.259259,0.00000,NaN
5,1,0004,20.0,592,59.3,615.548785,10.398375,14.335720,152.580131,55.809235,...,3.454188,0.714815,-80.802532,0.080385,26,0.268041,52.627583,78.949324,0.00000,NaN
6,1,0004,21.0,2193,236.6,3608.961429,15.487333,23.093233,236.190389,79.583285,...,9.109744,0.951440,-76.905548,0.058646,182,0.304858,54.151499,73.036936,0.00456,137.230784
7,1,0004,22.0,258,25.7,428.974103,16.691599,16.716387,88.937011,104.446205,...,4.375503,0.647222,146.388904,0.183244,52,0.565217,41.424793,78.186047,0.00000,NaN
8,1,0004,23.0,366,36.5,289.566214,7.933321,6.431369,48.812769,55.586874,...,2.631862,0.591803,88.008501,0.047680,20,0.454545,47.260960,79.251366,0.00000,NaN
9,1,0004,25.0,38,6.5,176.976793,34.757007,28.833878,134.868404,264.742029,...,12.647732,1.016667,-30.962087,0.104443,15,0.681818,45.380629,73.289474,0.00000,NaN
